In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import hydra
import torch
from omegaconf import DictConfig

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except Exception:
    print("Resolvers already registered:")

torch.set_float32_matmul_precision("high")

Resolvers already registered:


In [27]:
with hydra.initialize_config_dir(config_dir=str(BASE_DIR / "configs"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="base_chen",
        overrides=["trainer=debug", "model=cnn_chen_noise"],
    )

In [24]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()

Configuration hash: 1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c.pt.


In [28]:
model = hydra.utils.instantiate(
    cfg.model, rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None
)

InstantiationException: Error in call to target 'genpp.models.cgm.chen.chen.CNNChenNoiseModel':
TypeError('super(type, obj): obj (instance of CNNChenNoiseModel) is not an instance or subtype of type (_CNNChenModelBase).')
full_key: model

In [6]:
datamodule.setup(stage="fit")
dataloader = datamodule.val_dataloader()
batch = next(iter(dataloader))

In [20]:
res = model.forward(batch["x"], batch["timedelta"], n_samples=40)
print(res.shape)

ValueError: TD Scaling is not fitted yet.

In [21]:
# model.compile(mode="max-autotune")
logger = False
trainer = hydra.utils.instantiate(cfg.trainer, logger=logger, fast_dev_run=True)
trainer.fit(model, datamodule)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
/home/feik/GenPP/.pixi/envs/nb-gpu/lib/python3.13/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
Fitting fixed TD scaling: 100%|██████████| 171/171 [00:14<00:00, 11.79it/s]

  | Name                | Type            | Params | Mode  | FLOPs
------------------------------------------------------------------------
0 | final_activation    | FinalActivation | 0      | train | 0    
1 | loss_fn             | EnergyScore     | 0      | train | 0    
2 | embedding           | Embedding       | 6.4 K

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


In [47]:
# Try to load the model from the last checkpoint
ckpt_path = trainer.checkpoint_callback.best_model_path
# Get Model class
print(f"Loading model from checkpoint: {ckpt_path}")
model_loaded = model.__class__.load_from_checkpoint(ckpt_path, weights_only=False)

Loading model from checkpoint: /home/feik/GenPP/src/genpp/checkpoints/epoch=5-step=6.ckpt


In [48]:
trainer.predict(model_loaded, datamodule.val_dataloader(), return_predictions=False)

Predicting: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [49]:
datamodule.cleanup()